In [19]:
# Load packages

import pandas as pd
import numpy as np
import os

In [20]:
# Read CSV
df_2018 = pd.read_csv('../data_2018/18incd.csv')

df_2022 = pd.read_csv('../data_2023/22incdoh.csv')


df_2018.head(20)

,"OHIO, INDIANA, MICHIGAN, MISSOURI, PENNSYLVIANIA, WISCONSIN",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,Individual Income Tax Returns: \nSelected Inco...,Individual Income Tax Returns: \nSelected Inco...,NaN,NaN,NaN
1,[Money amounts are in thousands of dollars],[Money amounts are in thousands of dollars],NaN,NaN,NaN
2,State,Congressional\ndistrict [1],Size of adjusted gross income by congressional...,Number of returns,Percentage
3,NaN,NaN,NaN,NaN,NaN
4,Ohio,1,NaN,"358,550",100
5,Ohio,1,Under $1,"2,940",0.819969321
6,Ohio,1,"$1 under $10,000","47,370",13.21154651
7,Ohio,1,"$10,000 under $25,000","69,320",19.3334263
8,Ohio,1,"$25,000 under $50,000","84,850",23.66476084
9,Ohio,1,"$50,000 under $75,000","50,290",14.02593781


In [22]:
import pandas as pd

def clean_dataframe(df, separator=" "):
    """
    Cleans a pandas DataFrame by:
    1. Deleting the first 3 rows
    2. Dropping empty rows
    3. Combining the first two columns into one
    4. Renaming all columns
    
    Parameters:
        df (pd.DataFrame): Input DataFrame
        new_column_names (list): Optional list of new column names
        separator (str): String separator to combine first two columns (default = space)
    
    Returns:
        pd.DataFrame: Cleaned DataFrame
    """
    
    # 1. Drop first 3 rows
    df = df.iloc[3:].copy()
    
    # 2. Drop completely empty rows
    df = df.dropna(how="all")
    
    # 3. Combine first two columns into one
    state_col, district_col = df.columns[:2]
    df["CD116"] = (
            df[state_col].astype(str).str.strip()
            + " "
            + df[district_col].astype(int).astype(str).str.zfill(2)
        )
    cols = ["CD116"] + [c for c in df.columns if c != "CD116"]
    df = df[cols]
    df = df.drop(columns=[state_col, district_col, "Unnamed: 3"])

    # Rename columns
    new_column_names=["CD116", "Household Income", "Percentage"]
    df.columns = new_column_names
    df = df.dropna(subset=["Household Income"])

    # Change to wide format
    df_wide = df.pivot(index="CD116", columns="Household Income", values="Percentage").copy()
    df_wide = df_wide.reset_index()

    
    return df_wide

df_2018_clean = clean_dataframe(df_2018)

df_2022_clean = clean_dataframe(df_2022)
df_2022_clean = df_2022_clean.rename(columns={'CD116': 'CD119'})
df_2022_clean.head(20)

Household Income,CD119,"$1 under $10,000","$10,000 under $25,000","$100,000 under $200,000","$200,000 under $500,000","$25,000 under $50,000","$50,000 under $75,000","$500,000 or more","$75,000 under $100,000",Under $1
0,Ohio 01,10.42109037,16.41342415,16.46306152,6.364614069,23.02898271,15.03185065,1.5635772,9.574497422,1.138901911
1,Ohio 02,9.695713504,15.36402107,15.91386706,5.85542793,24.16731439,16.18447189,1.462417595,10.18222644,1.174540116
2,Ohio 03,10.38239692,21.03183081,9.885708182,2.197714164,29.83870968,16.66577654,0.440610981,8.50245674,1.054795984
3,Ohio 04,10.41397913,16.38911692,14.28403894,2.770611,26.37797584,17.81400258,0.501348657,10.33775067,1.111176264
4,Ohio 05,10.00449135,15.00112284,16.66573097,4.095553559,24.43858073,17.06433865,0.892656636,10.85223445,0.985290815
5,Ohio 06,10.53063907,17.98258584,13.87875801,2.720551996,26.12781337,16.54345326,0.614424183,10.36635453,1.235419747
6,Ohio 07,10.3349883,16.29287223,14.92609713,3.398390687,25.28676596,17.38857502,0.61633282,10.65171489,1.104262969
7,Ohio 08,10.12483063,15.98293309,16.25392798,4.116816098,25.04973045,16.48167902,0.683253092,10.2747427,1.032086949
8,Ohio 09,10.98064861,20.60283583,10.69528254,2.609910526,28.31069231,16.38476859,0.543979073,8.709610297,1.162272227
9,Ohio 10,10.23387586,17.38313434,15.32768639,3.9403313,25.22043306,16.09089069,0.699603943,10.00549276,1.098551646


In [23]:
df_2018_clean.to_csv('../data_2018/demographic_data/18incd_clean.csv')

df_2022_clean.to_csv('../data_2023/22incdoh_clean.csv')